# CODE 1.1: Check Raw Data

> **Goal:** see what is inside the original data, and make sure it is complete and correct before we use it.
>
> - **Input:** `data/raw/chewinggum/` (photos, masks and the photo list) and `data/raw/split_csv/` (VisA's own splits)
> - **Output:** no files. Only a tour of the data and a list of checks. At the end you should see **ALL CHECKS PASSED**.
>
> This notebook only reads the data. It never changes it.

## Import Libraries & Variables

### Import Libraries

> - `os`: works with folders and file paths.
> - `hashlib`: makes a "fingerprint" of a file, to find duplicate photos.
> - `numpy`: works with numbers. A mask is opened as a table of numbers.
> - `pandas`: reads and works with tables, like the photo list.
> - `PIL` (Pillow): opens photos and masks.
> - `tqdm`: shows a progress bar while the code goes through many photos.

In [1]:
import os
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

### Variables

#### Folder Paths

> This code finds the main project folder. It starts in the folder of this notebook and goes up until it finds the `data` folder, so the paths work wherever the notebook is.
>
> - `ROOT_DIR`: the main project folder.
> - `RAW_DIR`: the original data. We only read from it.
> - `GUM_DIR`: the chewing gum photos, masks and photo list.
> - `SPLIT_CSV_DIR`: VisA's own split files.
>
> If the data is missing, the code stops with a message that says what to do.

In [2]:
ROOT_DIR = os.getcwd()
while not os.path.isdir(os.path.join(ROOT_DIR, "data")) and ROOT_DIR != os.path.dirname(ROOT_DIR):
    ROOT_DIR = os.path.dirname(ROOT_DIR)

RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")
GUM_DIR = os.path.join(RAW_DIR, "chewinggum")
SPLIT_CSV_DIR = os.path.join(RAW_DIR, "split_csv")

if not os.path.exists(os.path.join(GUM_DIR, "image_anno.csv")):
    raise FileNotFoundError("Data not found. Copy the 'chewinggum' folder into data/raw/ (see README).")
if not os.path.exists(os.path.join(SPLIT_CSV_DIR, "1cls.csv")):
    raise FileNotFoundError("VisA split files not found in data/raw/split_csv/ (see README).")
print("Project folder found:", os.path.basename(ROOT_DIR))

Project folder found: visual-defect-inspection


#### Expected Values

> These values come from VisA's own documentation. The checks below test our copy of the data against them.
>
> - `EXPECTED_NORMAL` and `EXPECTED_DEFECT`: the table in [VisA's README](https://github.com/amazon-science/spot-diff) gives chewing gum 503 normal and 100 anomaly photos. *Anomaly* means "something unusual". Here it means a defect.
> - `VISA_MASK_CLASSES`: what each number in a chewing gum mask means, copied from VisA's file [`utils/id2class.py`](https://github.com/amazon-science/spot-diff/blob/main/utils/id2class.py). The README counts 6 anomaly classes for chewing gum: numbers 1 to 6.
> - `MASK_CODE`: the same list without 0 (background) and in lower case, because the photo list writes the names in lower case.
> - `PHOTO_ENDINGS`: the file endings that count as a photo.

In [3]:
EXPECTED_NORMAL = 503
EXPECTED_DEFECT = 100

VISA_MASK_CLASSES = {
    0: "BACKGROUND",
    1: "Chunk of gum missing",
    2: "Corner Missing",
    3: "Scratches",
    4: "Similar Colour Spot",
    5: "Small Cracks",
    6: "Other",
}

MASK_CODE = {number: name.lower() for number, name in VISA_MASK_CLASSES.items() if number != 0}

PHOTO_ENDINGS = {".jpg", ".jpeg", ".png"}

## Functions

### Show a Folder

> `show_folder()` prints a folder, its subfolders and the files inside: the names if there are 3 files or fewer, otherwise a count per file ending. It then does the same for every subfolder. `indent` is the space in front of the name, so subfolders are printed further to the right.

In [4]:
def show_folder(folder, indent):
    names = sorted(os.listdir(folder))
    files = [n for n in names if os.path.isfile(os.path.join(folder, n))]
    subfolders = [n for n in names if os.path.isdir(os.path.join(folder, n))]
    if len(files) <= 3:
        about = ", ".join(files)
    else:
        endings = pd.Series([os.path.splitext(n)[1].lower() for n in files]).value_counts()
        about = ", ".join(f"{count} {ending} files" for ending, count in endings.items())
    print(f"{indent}{os.path.basename(folder)}/" + (f"   ({about})" if about else ""))
    for sub in subfolders:
        show_folder(os.path.join(folder, sub), indent=indent + "    ")

### Record a Check

> `check()` prints PASS or FAIL for one check and stores the result in `results`, with the name of the check as the key. If a cell is run twice, the check is stored again under the same name, so it is never counted twice.

In [5]:
results = {}

def check(name, ok):
    results[name] = bool(ok)
    print(("PASS  " if ok else "FAIL  ") + name)

### List the Photos in a Folder

> `photo_files()` lists the photo files in one folder. Each path is written like in the photo list, for example `chewinggum/Data/Images/Normal/000.JPG`, so the two lists can be compared.

In [6]:
def photo_files(folder):
    return {os.path.relpath(os.path.join(folder, n), RAW_DIR).replace(os.sep, "/")
            for n in os.listdir(folder) if os.path.splitext(n)[1].lower() in PHOTO_ENDINGS}

### File Number

> `file_number()` gives the file name without its ending, for example `000` for `000.JPG`.

In [7]:
def file_number(rel_path):
    return os.path.splitext(os.path.basename(rel_path))[0]

### Photo Size

> `photo_size()` reads the width and height of a photo or mask, in pixels.

In [8]:
def photo_size(rel_path):
    with Image.open(os.path.join(RAW_DIR, rel_path)) as img:
        return img.size

### Numbers in a Mask

> `mask_numbers()` gives the numbers used in a mask, without 0 (0 means no defect).

In [9]:
def mask_numbers(rel_path):
    with Image.open(os.path.join(RAW_DIR, rel_path)) as mask:
        values = np.unique(np.asarray(mask))
    return {int(v) for v in values if v != 0}

### File Fingerprint

> `fingerprint()` makes an MD5 fingerprint of a file: a short code computed from every byte of the file. Two files with the same fingerprint have exactly the same content.

In [10]:
def fingerprint(rel_path):
    with open(os.path.join(RAW_DIR, rel_path), "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

## Look at the Raw Data

### The Folders

> VisA is shared as one folder per object (the objects are listed further below). We only downloaded the `chewinggum` folder. The `split_csv` folder holds three files from [VisA's GitHub](https://github.com/amazon-science/spot-diff/tree/main/split_csv) that say how VisA itself splits the photos into train and test.
>
> The code prints every folder inside `data/raw` and what it holds.

In [11]:
show_folder(RAW_DIR, indent="")

raw/
    chewinggum/   (image_anno.csv)
        Data/
            Images/
                Anomaly/   (100 .jpg files)
                Normal/   (503 .jpg files)
            Masks/
                Anomaly/   (100 .png files)
    split_csv/   (1cls.csv, 2cls_fewshot.csv, 2cls_highshot.csv)


> **What we see:**
>
> - `chewinggum/Data/Images/Normal/`: the photos of good gum (503 `.jpg` files).
> - `chewinggum/Data/Images/Anomaly/`: the photos of gum with a defect (100 `.jpg` files).
> - `chewinggum/Data/Masks/Anomaly/`: one mask per defect photo (100 `.png` files). A mask marks where the defect is (more in notebook 2.1). There is no `Masks/Normal` folder, because normal photos have nothing to mark.
> - `chewinggum/image_anno.csv`: the list of all photos.
> - `split_csv/`: VisA's own splits (`1cls.csv`, `2cls_fewshot.csv`, `2cls_highshot.csv`), explained in notebook 1.3.
>
> The checks below prove that every file belongs to the right photo.

### The Photo List

> `image_anno.csv` lists every photo. Each row is one photo:
>
> - `image`: where the photo file is
> - `label`: `normal`, or the defect type. A photo with more than one type has them split by commas.
> - `mask`: where the mask file is (empty for normal photos)
>
> We add one column, `is_defect`: `True` for a defect photo. The table shows the first 3 normal and the first 3 defect photos.

In [12]:
photos = pd.read_csv(os.path.join(GUM_DIR, "image_anno.csv")).fillna("")
photos["is_defect"] = photos["label"] != "normal"

print("Rows in the list:", len(photos))
photos.groupby("is_defect").head(3)

Rows in the list: 603


,image,label,mask,is_defect
0,chewinggum/Data/Images/Normal/000.JPG,normal,,False
1,chewinggum/Data/Images/Normal/001.JPG,normal,,False
2,chewinggum/Data/Images/Normal/002.JPG,normal,,False
503,chewinggum/Data/Images/Anomaly/000.JPG,"chunk of gum missing,scratches,small cracks",chewinggum/Data/Masks/Anomaly/000.png,True
504,chewinggum/Data/Images/Anomaly/001.JPG,"corner missing,small cracks",chewinggum/Data/Masks/Anomaly/001.png,True
505,chewinggum/Data/Images/Anomaly/002.JPG,"chunk of gum missing,small cracks",chewinggum/Data/Masks/Anomaly/002.png,True


### All VisA Objects

> Each file in `split_csv` lists every photo of the whole VisA dataset, not only chewing gum. So we can use one of them (`1cls.csv`) to see what the full dataset holds: how many normal and anomaly photos each object has.

In [13]:
visa_photos = pd.read_csv(os.path.join(SPLIT_CSV_DIR, "1cls.csv"))
objects = pd.crosstab(visa_photos["object"], visa_photos["label"])
objects["total"] = objects["normal"] + objects["anomaly"]

print(f"VisA objects: {len(objects)} | VisA photos: {objects['total'].sum()}")
objects.sort_values("total", ascending=False)

VisA objects: 12 | VisA photos: 10821


label,anomaly,normal,total
object,,,
pcb3,100,1006,1106
pcb4,100,1005,1105
pcb1,100,1004,1104
pcb2,100,1001,1101
macaroni1,100,1000,1100
macaroni2,100,1000,1100
candle,100,1000,1100
capsules,100,602,702
chewinggum,100,503,603


> **What we see:** VisA has 12 objects and 10,821 photos. We only use **chewinggum**: 503 normal and 100 anomaly photos, the same numbers as in VisA's README table. The other objects are not downloaded.

## Run the Checks

### Check the Photo List

> Is the list complete, and does it make sense? We also print which VisA classes the list really uses.

In [14]:
normal = photos[~photos["is_defect"]]
defect = photos[photos["is_defect"]]
types_in_list = set(defect["label"].str.split(",").explode())

print(f"Normal photos: {len(normal)} | Defect photos: {len(defect)}")
print("VisA classes used in the list:", sorted(types_in_list))
print("VisA classes never used:", sorted(set(MASK_CODE.values()) - types_in_list), "\n")

check("the list has the columns image, label and mask", {"image", "label", "mask"} <= set(photos.columns))
check(f"there are {EXPECTED_NORMAL} normal photos", len(normal) == EXPECTED_NORMAL)
check(f"there are {EXPECTED_DEFECT} defect photos", len(defect) == EXPECTED_DEFECT)
check("no photo is listed twice", photos["image"].is_unique)
check("every defect type in the list is a VisA class", types_in_list <= set(MASK_CODE.values()))
check("normal photos have no mask", (normal["mask"] == "").all())
check("every defect photo has a mask", (defect["mask"] != "").all())

Normal photos: 503 | Defect photos: 100
VisA classes used in the list: ['chunk of gum missing', 'corner missing', 'scratches', 'similar colour spot', 'small cracks']
VisA classes never used: ['other'] 

PASS  the list has the columns image, label and mask
PASS  there are 503 normal photos
PASS  there are 100 defect photos
PASS  no photo is listed twice
PASS  every defect type in the list is a VisA class
PASS  normal photos have no mask
PASS  every defect photo has a mask


### Check the Files

> Does every file in the list exist? Are there photos in the folders that are missing from the list? Is every file in the right place?

In [15]:
image_dir = os.path.join(GUM_DIR, "Data", "Images")
on_disk = photo_files(os.path.join(image_dir, "Normal")) | photo_files(os.path.join(image_dir, "Anomaly"))
same_number = defect["image"].map(file_number) == defect["mask"].map(file_number)

check("every photo in the list exists", all(os.path.exists(os.path.join(RAW_DIR, p)) for p in photos["image"]))
check("every mask in the list exists", all(os.path.exists(os.path.join(RAW_DIR, p)) for p in defect["mask"]))
check("no extra photos in the folders", on_disk == set(photos["image"]))
check("normal photos are in the Normal folder", normal["image"].str.contains("/Normal/").all())
check("defect photos are in the Anomaly folder", defect["image"].str.contains("/Anomaly/").all())
check("each mask has the same file number as its photo (like 000.JPG and 000.png)", same_number.all())

PASS  every photo in the list exists
PASS  every mask in the list exists
PASS  no extra photos in the folders
PASS  normal photos are in the Normal folder
PASS  defect photos are in the Anomaly folder
PASS  each mask has the same file number as its photo (like 000.JPG and 000.png)


### Check VisA's Split Files

> The split files must talk about the same photos as our photo list. For each file we check that it lists exactly our chewing gum photos, and that it marks the same photos as anomaly.

In [16]:
our_label = dict(zip(photos["image"], photos["is_defect"]))

for name in ["1cls", "2cls_highshot", "2cls_fewshot"]:
      visa = pd.read_csv(os.path.join(SPLIT_CSV_DIR, f"{name}.csv"))
      gum = visa[visa["object"] == "chewinggum"]
      check(f"{name}.csv lists exactly our {len(photos)} chewing gum photos",
            len(gum) == len(photos) and set(gum["image"]) == set(photos["image"]))
      check(f"{name}.csv marks the same photos as anomaly",
            ((gum["label"] == "anomaly") == gum["image"].map(our_label)).all())

PASS  1cls.csv lists exactly our 603 chewing gum photos
PASS  1cls.csv marks the same photos as anomaly
PASS  2cls_highshot.csv lists exactly our 603 chewing gum photos
PASS  2cls_highshot.csv marks the same photos as anomaly
PASS  2cls_fewshot.csv lists exactly our 603 chewing gum photos
PASS  2cls_fewshot.csv marks the same photos as anomaly


### Check Photo and Mask Sizes

> All photos should have the same size. Each mask must be exactly as big as its photo, or it would point to the wrong place.

In [17]:
tqdm.pandas(desc="Reading photo sizes")
photo_sizes = photos["image"].progress_map(photo_size)
tqdm.pandas(desc="Reading mask sizes")
mask_sizes = defect["mask"].progress_map(photo_size)
print("Photo sizes (width, height):", photo_sizes.value_counts().to_dict(), "\n")

check("all photos have the same size", photo_sizes.nunique() == 1)
check("each mask has the same size as its photo", (mask_sizes == photo_sizes[defect.index]).all())

Reading mask sizes: 100%|██████████| 100/100 [00:00<00:00, 136.80it/s]

Photo sizes (width, height): {(1342, 1118): 603} 

PASS  all photos have the same size
PASS  each mask has the same size as its photo


### Check the Mask Numbers

> A mask is a map of the defect. Each pixel holds a number: `0` = no defect, and the other numbers are the VisA classes in `MASK_CODE` (see Expected Values).
>
> For every defect photo we check that the mask is not empty, that it only uses VisA's numbers, and that its numbers give the same defect types as the photo list. The table counts how many masks use each number, next to how many photos the list gives that type.

In [18]:
masks_with_number = {number: 0 for number in MASK_CODE}
matches, empty_masks, unknown_numbers = 0, 0, 0

for _, row in tqdm(defect.iterrows(), total=len(defect), desc="Reading masks"):
    numbers = mask_numbers(row["mask"])
    if not numbers:
        empty_masks += 1
    if not numbers <= set(MASK_CODE):
        unknown_numbers += 1
    for n in numbers & set(MASK_CODE):
        masks_with_number[n] += 1
    types_from_mask = {MASK_CODE.get(n) for n in numbers}
    types_from_list = set(row["label"].split(","))
    if types_from_mask == types_from_list:
        matches += 1

print(f"Masks that match the list: {matches} of {len(defect)}\n")
check("every mask has at least one defect pixel", empty_masks == 0)
check("masks only use VisA's numbers", unknown_numbers == 0)
check("mask numbers match the defect types in the list", matches == len(defect))

photos_per_type = defect["label"].str.split(",").explode().value_counts()
mask_table = pd.DataFrame({"VisA class": MASK_CODE, "masks with this number": masks_with_number})
mask_table["photos with this type in the list"] = mask_table["VisA class"].map(photos_per_type).fillna(0).astype(int)
mask_table.rename_axis("number in mask")

Reading masks: 100%|██████████| 100/100 [00:00<00:00, 126.40it/s]

Masks that match the list: 100 of 100

PASS  every mask has at least one defect pixel
PASS  masks only use VisA's numbers
PASS  mask numbers match the defect types in the list


,VisA class,masks with this number,photos with this type in the list
number in mask,,,
1,chunk of gum missing,47,47
2,corner missing,36,36
3,scratches,16,16
4,similar colour spot,25,25
5,small cracks,29,29
6,other,0,0


> **What we see:** for every number, the count of masks is the same as the count of photos with that type in the list. For example, 47 masks use number 1 and 47 photos are labeled *chunk of gum missing*. Number 6 (*other*) is never used. So the README counts 6 classes for chewing gum, but the real data only has 5 defect types.

### Check for Exact Copies

> If the same photo is in the data twice, one copy could land in train and the other in test. Then the test would be too easy. This is called **data leakage**.
>
> Two files with the same fingerprint are exact copies. This only finds exact copies. If the same gum was photographed twice, the two files are different (the gum moves a little, the light changes), so they are not found here. Notebook 1.2 looks for those.

In [19]:
tqdm.pandas(desc="Making fingerprints")
fingerprints = photos["image"].progress_map(fingerprint)
n_copies = int(fingerprints.duplicated().sum())
print("Exact copies found:", n_copies, "\n")

check("no photo is an exact copy of another", n_copies == 0)

Making fingerprints: 100%|██████████| 603/603 [00:00<00:00, 2367.68it/s]

Exact copies found: 0 

PASS  no photo is an exact copy of another


## Result

### Final Result

> If every check passed, the data is ready for the next notebook. If a check failed, its name is printed.

In [20]:
failed = [name for name, ok in results.items() if not ok]
if results and not failed:
    print(f"ALL CHECKS PASSED ({len(results)} checks)")
else:
    print(f"{len(failed)} of {len(results)} checks FAILED:")
    for name in failed:
        print("  -", name)

ALL CHECKS PASSED (25 checks)


## Summary

> - The data has the folders VisA describes: 503 normal photos, 100 defect photos, 100 masks and the photo list.
> - Every file exists, sits in the right folder and has the right size. Every mask has the same file number as its photo.
> - VisA's three split files list exactly our chewing gum photos, with the same labels.
> - The mask numbers follow VisA's class list. Numbers 1 to 5 are used, and 6 (*other*) never appears.
> - No photo is an exact copy of another. Photos of the same gum taken twice are checked in the next notebook.
>
> **Next:** `1.2_find_same_gum_photos.ipynb`